## WSW Gates - Beach Courses

Generate beach courses for Weymouth Speed Week

In [1]:
import os
import sys

import jinja2

import pyproj

### Constants

Coordinates of South West Coast Path waypoints were identified on Google Earth

In [2]:
# OTC
WAYPOINT_1 = (-2.46500000, 50.57482500)

# Billy Winters
WAYPOINT_2 = (-2.46826944, 50.57890556)

In [3]:
# Course length allows for GPS inaccuracy / buoys being slightly mislaid
COURSE_LENGTH = 520

# Course is slightly shifted towards Chesil
COURSE_SHIFT = 10

# Course extends well into the hrbouar for foils
COURSE_WIDTH = 1500

# Azimuth is parallel to the road
COURSE_AZIMUTH = 330

### Jinja Setup

Prepare environment for Jinja templates

In [4]:
projdir = os.path.realpath(os.path.join(sys.path[0], '..'))

coursesPath = os.path.join(projdir, 'courses')

gtxPath = os.path.join(coursesPath, 'gtx')
kmlPath = os.path.join(coursesPath, 'kml')

templateLoader = jinja2.FileSystemLoader([gtxPath, kmlPath])

templateEnv = jinja2.Environment(loader=templateLoader,
                                 autoescape=True, trim_blocks=True, lstrip_blocks=True)

### Calculate distance and azimuths using pyproj

The pyproj library returns distances in metres, and azimuths betwen -180 and +180

In [5]:
geod = pyproj.Geod(ellps='WGS84')

In [6]:
# Use SWCP waypoints
lon1, lat1 = WAYPOINT_1
lon2, lat2 = WAYPOINT_2

# Determine forward and back azimuths, plus distance between the waypoints
forward_azimuth, back_azimuth, distance = geod.inv(lon1, lat1, lon2, lat2)

# Convert negative values to positive values
forward_azimuth = (forward_azimuth + 360) % 360
back_azimuth = (back_azimuth + 360) % 360
line_azimuth = (forward_azimuth + 90)  % 360

# Report the results
print(f"Distance: {distance:.3f} meters")
print(f"Heading: {forward_azimuth:.3f} degrees")

Distance: 509.587 meters
Heading: 332.971 degrees


### Calculate Points for Start / Finish Line

Simple forward projection

In [7]:
def calculatePoints(waypoints, i, line_azimuth, gate_width):
    '''Get corners'''

    c1_lon = waypoints.lons[i]
    c1_lat = waypoints.lats[i]
    c2_lon, c2_lat, ignore = geod.fwd(c1_lon, c1_lat, line_azimuth, gate_width)
    mid_lon, mid_lat, ignore = geod.fwd(c1_lon, c1_lat, line_azimuth, gate_width / 2)

    return c1_lon, c1_lat, c2_lon, c2_lat, mid_lon, mid_lat

### Generate Gate File

Use Jinja to generate .gtx file from template

In [8]:
def saveGtx(w1, w2, i, line_azimuth):
    '''Save GTX for GPSResults'''
    
    track_length = -500
    gate_width = 1500

    # Calculate corners, and start + end points
    
    c1_lon, c1_lat, c2_lon, c2_lat, start_lon, start_lat = calculatePoints(w1, i, line_azimuth, gate_width)   
    c3_lon, c3_lat, c4_lon, c4_lat, finish_lon, finish_lat = calculatePoints(w2, i, line_azimuth, gate_width)
    
    # Save Gate XML
    
    corners_lat_lon = "{:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f} {:.7f}".format(
        c1_lat, c1_lon, c2_lat, c2_lon, c3_lat, c3_lon, c4_lat, c4_lon)
    
    template = templateEnv.get_template("template.gtx")
    gtx = template.render(track_length=track_length, gate_width=gate_width,
                          start_lat=round(start_lat, 7), start_lon=round(start_lon, 7),
                          finish_lon=round(finish_lon, 7), finish_lat=round(finish_lat, 7),
                          corners_lat_lon=corners_lat_lon
                         )
    
    gtxFile = os.path.join(gtxPath, 'test.gtx')
    with open(gtxFile, 'w', encoding='utf-8') as f:
    	f.write(gtx)

### Common Functions for GTX and KML

Calculation for corners, start + finish, etc

In [9]:
def calculateCorners(mid_lon, mid_lat, azimuth, distance, shift, gate_width):
    '''Calculate corners for course'''

    # c1 = start line (shore), c3 = finish line (shore)
    c1 = geod.fwd(mid_lon, mid_lat, (azimuth + 180) % 360, distance / 2)
    c3 = geod.fwd(mid_lon, mid_lat, azimuth, distance / 2)
    
    # Move c1 + c3 towards chesil by 25 meters
    c1 = geod.fwd(c1[0], c1[1], (azimuth - 90) % 360, shift)
    c3 = geod.fwd(c3[0], c3[1], (azimuth - 90) % 360, shift)

    # c2 = start line (harbour), c4 = finish line (harbour)
    c2 = geod.fwd(c1[0], c1[1], (azimuth + 90) % 360, gate_width)
    c4 = geod.fwd(c3[0], c3[1], (azimuth + 90) % 360, gate_width)

    return c1, c2, c3, c4


def calculateStartFinish(c1, c2, c3, c4, azimuth, gate_width):
    '''Calculate start and finish points for course'''

    # Points at the middle of the start and finish lines
    start = geod.fwd(c1[0], c1[1], (azimuth + 90) % 360, gate_width / 2)
    finish = geod.fwd(c3[0], c3[1], (azimuth + 90) % 360, gate_width / 2)

    return start, finish

### Generate KML File

Use Jinja to generate .kml file from template

In [10]:
def saveKml(mid_lon, mid_lat):
    '''Save KML for Google Earth'''
    
    # c1 + c2 = start line, c3 + c4 = finish line
    c1, c2, c3, c4 = calculateCorners(mid_lon, mid_lat, COURSE_AZIMUTH, COURSE_LENGTH, COURSE_SHIFT, COURSE_WIDTH)

    # Start and finish points are at the middle of each line
    start, finish = calculateStartFinish(c1, c2, c3, c4, COURSE_AZIMUTH, COURSE_WIDTH)
    
    # Save KML
    
    name = 'Weymouth Speed Week'
    
    polygon_coordinates = \
        "{:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0 {:.7f},{:.7f},0".format(
        c1[0], c1[1], c3[0], c3[1], c4[0], c4[1], c2[0], c2[1], c1[0], c1[1])
    
    template = templateEnv.get_template("template.kml")
    kml = template.render(name=name,
                          start_lon=round(start[0], 7), start_lat=round(start[1], 7),
                          finish_lon=round(finish[0], 7), finish_lat=round(finish[1], 7),
                          polygon_coordinates=polygon_coordinates
                         )
    
    kmlFile = os.path.join(kmlPath, 'test.kml')
    with open(kmlFile, 'w', encoding='utf-8') as f:
    	f.write(kml)

    print("z")

### Generate Multiple Courses

Typically 50 meter intervals

In [11]:
interval = 50

w1 = geod.fwd_intermediate(lon1, lat1, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)
w2 = geod.fwd_intermediate(lon2, lat2, back_azimuth, npts=6, del_s=interval, initial_idx=0,
                           return_back_azimuth=True)

In [12]:
i = 3

mid_lon = (w1.lons[i] + w2.lons[i]) / 2
mid_lat = (w1.lats[i] + w2.lats[i]) / 2

saveGtx(w1, w2, i, line_azimuth)
saveKml(mid_lon, mid_lat)

z
